# Pipeline demo — all standardized theories

Runs every theory in `pipeline.theories.*` through the pipeline
(`compute_cumulants` + `generate_report`), so the user can verify
the framework is healthy without bringing in any simulator-side
machinery.  This notebook is **theory-only**: no Hawkes simulator,
no `cumulant_estimator`, no run-vs-sim comparison.  Those live
alongside the model-specific simulator files in `models/` and
are not part of the `pipeline/` package.

Each theory:
* loads its `HAWKES_MODEL` dict (assembled from `TheoryBuilder` +
  templates),
* runs `pipeline.compute_cumulants` end-to-end,
* drops a `.npz` of the τ-grid evaluation under `pipeline_outputs/`,
* drops a multi-page PDF report there too,
* renders an inline plot of $C^{(2)}(\tau)$ for at-a-glance comparison.


## 1. Setup

In [ ]:
import sys, os
import os, sys
# --- depth-robust repo root: walk up until we find the 'pipeline' package ---
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'pipeline')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(os.path.join(_root, 'notebooks'))  # cwd=notebooks/ so relative data paths resolve as before

import importlib
import numpy as np
import matplotlib.pyplot as plt

from pipeline import compute_cumulants, generate_report

# Where to drop NPZ + PDF artifacts (gitignored).
OUT_DIR = '../pipeline_outputs/demo'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output dir: {os.path.abspath(OUT_DIR)}')


## 2. Configuration

One `fundamental` dict good for all 5 theories.  Unused entries
(e.g. `a` for the unit-gain delta-synapse model, or the GTaS
parameters for non-GTaS models) are simply ignored by
`compute_cumulants` — only parameters declared in the model dict
are picked up.


In [ ]:
fundamental = {
    'E':                   [0.5, 0.5],
    'w':                   [[0.3, 0.25], [0.3, 0.35]],
    'tau':                 10.0,
    'a':                   0.44,
    'tau_g':               2.5,
    # GTaS-only parameters — ignored by non-GTaS theories
    'w_X':                 0.25,
    'lambda_X':            1.7,
    'p_part':              0.6,
    'mu_shift_diff':       0.0,
    'sigma_shift_diff_sq': 1.0,
}

K       = 2                      # cumulant order
MAX_ELL = 0                      # tree-only for the demo (1-loop is
                                 # straightforward but multiplies
                                 # runtime by ~10x).
EXTERNAL_FIELDS = [('dn', 1), ('dn', 2)]
TAU_MAX  = 50.0
TAU_STEP = 0.5
print(f'k = {K},  max_ell = {MAX_ELL},  '
      f'external_fields = {EXTERNAL_FIELDS}')


## 3. List of theories to run

Each entry: a short label + the import path of the standardized
theory module.  Each module exposes a single `HAWKES_MODEL` dict
built via `pipeline.theory.TheoryBuilder` + the templates in
`pipeline.theory_templates`.


In [ ]:
THEORIES = [
    ('linear_delta',      'pipeline.theories.linear_hawkes_2pop_delta'),
    ('linear_expg',       'pipeline.theories.linear_hawkes_2pop_expg'),
    ('quad_expg',         'pipeline.theories.quad_hawkes_2pop_expg'),
    ('linear_expg_gtas',  'pipeline.theories.linear_hawkes_2pop_expg_gtas'),
    ('quad_expg_gtas',    'pipeline.theories.quad_hawkes_2pop_expg_gtas'),
]
for label, mod_path in THEORIES:
    mod = importlib.import_module(mod_path)
    print(f'  {label:<22}  {mod.HAWKES_MODEL["name"]}')


## 4. Run the pipeline on each theory

End-to-end: FieldTheory.expand → propagator build → MF solve →
diagram enumeration → Phase J integration on a τ grid.  Each call
saves a `.npz` to `pipeline_outputs/demo/<label>.npz` and stashes
the `result` dict in `RESULTS[label]` for downstream use.

Skipped: simulation, cumulant estimation from spike trains,
sim/theory overlays — all out of `pipeline/`'s scope.


In [ ]:
RESULTS = {}
for label, mod_path in THEORIES:
    print(f'\n{"="*70}\n  {label}\n{"="*70}')
    mod = importlib.import_module(mod_path)
    RESULTS[label] = compute_cumulants(
        model           = mod.HAWKES_MODEL,
        k               = K,
        max_ell         = MAX_ELL,
        fundamental     = fundamental,
        external_fields = EXTERNAL_FIELDS,
        tau_max         = TAU_MAX,
        tau_step        = TAU_STEP,
        output_npz      = f'{OUT_DIR}/{label}.npz',
        verbose         = True,
    )


## 5. Side-by-side $C^{(2)}(\tau)$ comparison

All 5 theories on one axes.  Same external fields, same parameters.
Differences encode: phi (linear vs quadratic), synaptic kernel
(delta vs exp), GTaS noise on/off.  Useful sanity check that
(a) every theory produced a sane curve and (b) the structure of
the differences is what you'd expect from the action.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, _ in THEORIES:
    r = RESULTS[label]
    ax.plot(r['tau_grid'], r['C_tau'].real, linewidth=1.6,
            label=label)
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.6)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.6)
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'$C^{(2)}(\tau)$')
ax.set_title(rf'$C^{{({K})}}(\tau)$ for {EXTERNAL_FIELDS}, '
             f'tree-level (max_ell={MAX_ELL})')
ax.legend(fontsize=9, loc='best')
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()


## 6. Summary table

Mean-field rates, source counts, and integration grid stats
for each theory.


In [ ]:
rows = []
for label, _ in THEORIES:
    r = RESULTS[label]
    mf = r['mf_values']
    n_diag    = len(r['diagrams'])
    nstar_str = ', '.join(f'{x:.4f}' for x in mf['nstar'])
    vstar_str = ', '.join(f'{x:.4f}' for x in mf['vstar'])
    # mstar is GTaS-only — non-GTaS theories don't carry the key
    # (compute_cumulants drops MF params it never solved), so reach for
    # it with .get() instead of assuming it is always present.
    mstar = mf.get('mstar')
    mstar_str = (', '.join(f'{x:.3f}' for x in mstar)
                 if mstar is not None else '—')
    n_poles = (len(r['propagator']['pole_vals'])
               if r['propagator']['pole_vals'] else 0)
    rows.append([label, n_diag, n_poles, nstar_str, vstar_str, mstar_str])

header = ['theory', 'diagrams', 'poles', 'n*', 'v*', 'm* (b_X)']
widths = [max(len(str(r[i])) for r in [header] + rows)
          for i in range(len(header))]
fmt = '  '.join(f'{{:<{w}}}' for w in widths)
print(fmt.format(*header))
print(fmt.format(*['-'*w for w in widths]))
for r in rows:
    print(fmt.format(*[str(x) for x in r]))


## 7. Generate per-theory PDF reports

Each report is a multi-page PDF: cover (model, parameters, MF
values, total slice plot) + one page per typed diagram (graph,
vertex assignments, contribution plot).  Useful for scrolling
through and gaining intuition.  Reuses the `result` dict from
above so we don't re-run the heavy compute.


In [ ]:
for label, mod_path in THEORIES:
    mod = importlib.import_module(mod_path)
    pdf_path = f'{OUT_DIR}/{label}_report.pdf'
    print(f'\n  → {pdf_path}')
    generate_report(
        model           = mod.HAWKES_MODEL,
        k               = K,
        fundamental     = fundamental,
        external_fields = EXTERNAL_FIELDS,
        output_pdf      = pdf_path,
        result          = RESULTS[label],   # reuse compute
        verbose         = False,
    )
print('\nAll reports written.')


## 8. (Optional) Spot-check one diagram

Pull the typed-diagram list from any result dict and inspect
vertex types / classify info / per-diagram contribution callable.
This is what `generate_report` walks page-by-page.


In [ ]:
label = 'quad_expg_gtas'
r = RESULTS[label]
for idx, td_record in enumerate(r['diagrams']):
    td   = td_record['typed_diagram']
    info = td_record['classify']
    pf   = td_record['combined_prefactor']
    print(f'\n--- diagram {idx} ---')
    # 'Scal' is the per-diagram symmetry / combinatorial factor 𝒮(Γ)
    # in the classify record (was 'M' in an earlier API).
    print(f'  Scal (combinatorial) = {info.get("Scal")}')
    print(f'  scalar prefactor     = {pf}')
    for v, vt in sorted(td.vertex_assignments.items()):
        cls = type(vt).__name__
        rl  = getattr(vt, "response_legs", [])
        pl  = getattr(vt, "physical_legs", None)
        s   = f'    v{v} ({cls}): resp={rl}'
        if pl is not None: s += f', phys={pl}'
        print(s)


## Summary

A **theory-only batch driver**: it runs every standardized theory in `pipeline.theories.*`
through `compute_cumulants` + `generate_report`, drops a `.npz` and a PDF report per theory under
`pipeline_outputs/`, and renders a side-by-side $C^{(2)}(\tau)$ comparison plus a summary table —
a quick health check of the framework with no simulator-side machinery.

**Knobs:**
* **`fundamental`** — shared parameters fed to every theory (`E`, `w`, `tau`, `a`, `tau_g`,
  plus GTaS-only extras that non-GTaS theories ignore).
* **`K`, `MAX_ELL`** — cumulant order and loop order for the batch (tree-only by default; 1-loop
  multiplies runtime by ~10×).
* **`EXTERNAL_FIELDS`** — the two legs used in the comparison plot.
* **`TAU_MAX`, `TAU_STEP`** — the $\tau$ grid.
* **`THEORIES`** — the list of `(label, module path)` theories to run.